In [11]:
import pandas as pd

df = pd.read_parquet("data/prepared_recipes.parquet")

df.head()

,name,ingredients,description,steps
0,roast butternut squash with maple syrup and gi...,"[butternut squash, ghee, maple syrup, pine nut...","found in times2, made it several times since w...","[preheat the oven to 350f, peel the squash , c..."
1,blond brownies with br sugar frosting,"[granulated sugar, vanilla, butter, eggs, flou...",have not tried these yet..but want to some day...,"[heat oven to 350 --, beat sugars , butter , v..."
2,hershey s double chocolate and peanut butter c...,"[butter, vanilla, cocoa, salt, nuts, sugar, eg...",chocolate cookies with chocolate and peanut bu...,"[preheat oven to 350 degrees, in a large mixin..."
3,latte frozen yogurt,"[sugar, cornstarch, low-fat milk, instant coff...",the flavor on this is absolutely amazing! mor...,"[in a 2-quart saucepan , combine sugar , insta..."
4,cinnamon quick bread,"[vegetable oil, egg, salt, cinnamon, sugar, bu...",yummy and easy! good with a cup of tea in the...,"[filling: mix and set aside, mix flour , bakin..."


In [12]:
from transformers import AutoTokenizer, AutoModel
from tqdm import tqdm
tqdm.pandas()

model = AutoModel.from_pretrained("google/bert_uncased_L-2_H-128_A-2")
tokenizer = AutoTokenizer.from_pretrained("google/bert_uncased_L-2_H-128_A-2")

In [13]:
from vicinity import Vicinity
vicinity = Vicinity.load("data/vicinity_recipe_vectors")

In [14]:
def embed(text):
	if not isinstance(text, str):
		return None
	inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True, max_length=512)
	outputs = model(**inputs)
	return outputs.pooler_output.squeeze().detach().numpy()


# query the index
ingredient = "apple"
prompt = f"Write a new recipe that uses the following ingredient: {ingredient}."

query_vector = embed(prompt)
query_vector

array([-0.99999   , -0.11441393, -0.99933255,  0.8717775 , -0.99752414,
        0.4510298 , -0.8857669 , -0.8380687 ,  0.01503762, -0.07621382,
       -0.59485006, -0.13093674,  0.01318978,  0.9999736 , -0.12814279,
       -0.9658958 ,  0.7662259 ,  0.17728746, -0.8567302 , -0.11801536,
        0.9308745 , -0.12901324,  0.65019286, -0.6874064 , -0.9995216 ,
       -0.00239357, -0.99912363,  0.92720497,  0.9168838 ,  0.05736577,
       -0.01041014, -0.10968667, -0.99761504, -0.8943712 ,  0.883577  ,
        0.99990183, -0.96026427,  0.17850025,  0.9215124 , -0.99507076,
        0.96860194,  0.944545  , -0.99966407,  0.9725057 , -0.9989432 ,
        0.01494563, -0.9986061 ,  0.99944764,  0.7560421 ,  0.9912517 ,
        0.9331573 , -0.86229664, -0.15938313,  0.99671334,  0.9066384 ,
        0.9941897 , -0.97805315, -0.36131167,  0.9556035 , -0.4224399 ,
        0.14218566,  0.49319357,  0.54087186,  0.90510243,  0.24349979,
       -0.99996   , -0.8131223 ,  0.00653521,  0.92180574,  0.91

In [ ]:
results = vicinity.query(
	query_vector,
	k=3,
)

# results = vicinity.query_threshold(
# 	query_vector,
# 	threshold=0.9,
# )

# print the results
for result in results:
	for item in result:
		print(f"Text: {item[0]}")
		# item[0] = distance to query vector
		print(f"Distance: {item[1]}")
		print(f"Sim: {1.0 - item[1]}")
		print()

Text: the world s easiest chocolate mousse
chocolate bars, milk, honey, fresh cream
place milk in a pan
-place chocolate in it and melt on a medium flame
-add honey
-remove from flame , add fresh cream and mix well
-whip it for 3-5 minutes with an electronic egg beater
-cool it
-then transfer to a refridgerator
-the mousse is ready after 2 hours
Distance: 0.022306084632873535
Sim: 0.9776939153671265

Text: feijoa smoothie
feijoas, milk, vanilla ice cream, honey
cut feijoas in half and scoop out the flesh , put in to blender
-add remaining ingredients and blend until frothy
Distance: 0.023579180240631104
Sim: 0.9764208197593689

Text: dairy queen blizzard
vanilla ice cream, sugar, whipping cream, fudge sauce, candy bars
combine all ingredients in a blender
-blend for 20-30 seconds on medium speed
-stop the blender and stir with a long spoon
-repeat until well blended and smooth
-pour into a large glass
-at this point you may place in the freezer for until desired consistency is achieved

In [16]:
# prepare the model input
example_recipies = "\n\n ".join([result[0][0] for result in results])
prompt_with_rag = f"{prompt} Use the following information to help you: {example_recipies}"
prompt_with_rag

'Write a new recipe that uses the following ingredient: apple. Use the following information to help you: the world s easiest chocolate mousse\nchocolate bars, milk, honey, fresh cream\nplace milk in a pan\n-place chocolate in it and melt on a medium flame\n-add honey\n-remove from flame , add fresh cream and mix well\n-whip it for 3-5 minutes with an electronic egg beater\n-cool it\n-then transfer to a refridgerator\n-the mousse is ready after 2 hours'

In [17]:
from transformers import AutoModelForCausalLM, AutoTokenizer

# https://arxiv.org/pdf/2505.09388
model_name = "Qwen/Qwen3-0.6B"

# load the tokenizer and the model
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)

In [18]:
def generate(llm_model: AutoModel, prompt_text: str):
    # structure for chat completion
    messages = [
        {"role": "user", "content": prompt_text}
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False # Switches between thinking and non-thinking modes. Default is True.
    )
    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

    # conduct text completion
    generated_ids = llm_model.generate(
        **model_inputs,
        max_new_tokens=32768
    )
    output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist() 

    # parsing thinking content
    try:
        # rindex finding 151668 (</think>)
        index = len(output_ids) - output_ids[::-1].index(151668)
    except ValueError:
        index = 0

    thinking_content = tokenizer.decode(output_ids[:index], skip_special_tokens=True).strip("\n")
    content = tokenizer.decode(output_ids[index:], skip_special_tokens=True).strip("\n")
    return thinking_content, content

In [19]:
# generate just using the LLM
thinking_content, content = generate(model, prompt)
print("content:", content)

content: **Apple Cider Vinegar Smoothie Recipe**

**Ingredients:**
- 1 cup fresh apple juice (or apple cider)
- 1/2 cup apple cider vinegar
- 1/4 cup water
- 1/2 cup raw honey
- 1/4 cup almond milk (or coconut milk for a vegan option)
- 1/2 cup chopped fresh spinach
- 1/4 cup chopped walnuts (optional)
- 1/2 cup chopped banana slices

**Instructions:**
1. In a large bowl, combine the apple juice, apple cider vinegar, water, honey, almond milk, spinach, and banana slices.
2. Stir until everything is well combined.
3. Serve warm and enjoy!

**Notes:**
- For a tangy flavor, add a splash of lemon juice at the end.
- For a vegan version, use coconut milk instead of almond milk.

Enjoy your apple cider vinegar smoothie! 🍎✨


In [20]:
# generate with RAG
thinking_content, content = generate(model, prompt_with_rag)
print("content:", content)

content: Here's a **simple and easy recipe** using **apple** and the **world's easiest chocolate mousse** method:

---

**World's Easiest Chocolate Mousse Recipe**

**Ingredients:**

- 2 cups milk  
- 1 cup chocolate bars  
- 1/4 cup honey  
- 1/2 cup fresh cream  
- 1/2 cup apple (or 1 cup of pure apple juice, if you want a sweet twist)  
- 1/2 egg (for whipping)  
- Optional: a few drops of vanilla extract

---

**Steps:**

1. **Melt the chocolate:** Place the chocolate bars in a pan, add milk, and melt on medium heat.  
2. **Add honey:** Once the chocolate is melted, stir in the honey.  
3. **Add cream:** Remove the flame, add fresh cream, and mix thoroughly.  
4. **Whip the mixture:** Whip the chocolate mixture using an electronic egg beater for **3–5 minutes**.  
5. **Cool it:** Let the mousse cool to room temperature.  
6. **Transfer to the fridge:** Transfer the mousse to a sealed container and store in the fridge for **2 hours**.  
7. **Serve:** Serve warm or cold, and enjoy!

